[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

# Laboratorio 5: Modelos en PyTorch
## Parte 2: Regresión Logística Binaria (Clasificación)
En esta segunda libreta entrenaremos un dataset utilizando Regresión Logística (Binaria). Usaremos el dataset de "Buzz in Social Media" para determinar si un tema será popular o no (Clase 1 o Clase 0).

In [ ]:
# IMPORTANTE: Si estás ejecutando este cuadernillo en Google Colab, 
# descomenta y ejecuta estas líneas para montar tu Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

### 2.1 Preprocesamiento: Tarea continua a Clasificación Binaria
Convertiremos el objetivo continuo (promedio de discusiones) en 0 y 1 para predecir Popularidad.

In [ ]:
archivo_buzz = 'Twitter.data'
df_buzz = pd.read_csv(archivo_buzz, header=None)

X_buzz_pandas = df_buzz.iloc[:, :-1].values 
Y_buzz_pandas_raw = df_buzz.iloc[:, -1].values  

# Convertimos el target en Clases (0 y 1) cortando por el promedio global.
umbral = Y_buzz_pandas_raw.mean()
Y_buzz_clases = (Y_buzz_pandas_raw > umbral).astype(int)

m = len(Y_buzz_clases)
print(f"Total de ejemplos (m): {m} ")
print(f"Total de propiedades (n): {X_buzz_pandas.shape[1]} ")

unicos, conteos = np.unique(Y_buzz_clases, return_counts=True)
print(f"Clase 0 (No Popular): {conteos[0]}")
print(f"Clase 1 (Popular): {conteos[1]}")

### 2.2 Separación del 80/20, Normalización y Dataset

In [ ]:
limite_buzz = int(0.8 * m)

X_train_buzz_raw = X_buzz_pandas[:limite_buzz]
X_test_buzz_raw = X_buzz_pandas[limite_buzz:]
Y_train_buzz = Y_buzz_clases[:limite_buzz]
Y_test_buzz = Y_buzz_clases[limite_buzz:]

X_mean_buzz = X_train_buzz_raw.mean(axis=0)
X_std_buzz = X_train_buzz_raw.std(axis=0) + 1e-8

X_train_buzz = (X_train_buzz_raw - X_mean_buzz) / X_std_buzz
X_test_buzz = (X_test_buzz_raw - X_mean_buzz) / X_std_buzz

y_train_buzz = Y_train_buzz.astype(np.float32).reshape(-1, 1)
y_test_buzz = Y_test_buzz.astype(np.float32).reshape(-1, 1)
X_train_buzz = X_train_buzz.astype(np.float32)
X_test_buzz = X_test_buzz.astype(np.float32)

class DatasetBinario(torch.utils.data.Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float().to(device)
        self.Y = torch.from_numpy(Y).float().to(device) 
    def __len__(self):
        return len(self.X)
    def __getitem__(self, ix):
        return self.X[ix], self.Y[ix]

dataset_train_bin = DatasetBinario(X_train_buzz, y_train_buzz)
dataset_test_bin = DatasetBinario(X_test_buzz, y_test_buzz)

batch_size_bin = 2000
dataloader_train_bin = torch.utils.data.DataLoader(dataset_train_bin, batch_size=batch_size_bin, shuffle=True)
dataloader_test_bin = torch.utils.data.DataLoader(dataset_test_bin, batch_size=batch_size_bin, shuffle=False)
print("DataLoaders listos.")

### 2.3 Diseño de la Red y el Entranamiento
Uso de Función Sigmoide en la última capa para Regresión Logística y `BCELoss`.

In [ ]:
D_in_bin = X_train_buzz.shape[1] 
H_bin = 150 
D_out_bin = 1 

model_bin = torch.nn.Sequential(
    torch.nn.Linear(D_in_bin, H_bin),
    torch.nn.ReLU(),
    torch.nn.Linear(H_bin, D_out_bin),
    torch.nn.Sigmoid() 
).to(device)

criterion_bin = torch.nn.BCELoss() 
optimizer_bin = torch.optim.Adam(model_bin.parameters(), lr=0.005)

epochs_bin = 250 
log_each_bin = 50
l_bin = []
model_bin.train()

print("Iniciando Entrenamiento...")
for e in range(1, epochs_bin + 1):
    _l = []
    for x_b, y_b in dataloader_train_bin:
        y_pred = model_bin(x_b)
        loss = criterion_bin(y_pred, y_b) 
        
        _l.append(loss.item())
        optimizer_bin.zero_grad()
        loss.backward()
        optimizer_bin.step()

    l_bin.append(np.mean(_l))
    if not e % log_each_bin:
        print(f"Epoch {e}/{epochs_bin} - Costo (BCELoss): {np.mean(l_bin):.5f}")

PATH_BIN = './checkpoint_clasificacion_binaria.pt'
torch.save(model_bin.state_dict(), PATH_BIN)
print(f"Modelo guardado en: {PATH_BIN}")

plt.plot(range(1, epochs_bin + 1), l_bin, color='green')
plt.title('Costo - Regresión Logística Binaria')
plt.xlabel('Épocas')
plt.ylabel('Costo (BCELoss)')
plt.grid(True)
plt.show()

### 2.4 Precisión (Accuracy) y Ejemplos de prueba

In [ ]:
model_bin.eval() 

correctos = 0
total_ejemplos = len(dataset_test_bin)

with torch.no_grad():
    for x_b, y_b in dataloader_test_bin:
        probabilidades = model_bin(x_b)
        
        # Forzamos probabilidad a clase definida (> 0.5)
        prediccion_clase = (probabilidades >= 0.5).float()
        correctos += (prediccion_clase == y_b).sum().item()

accuracy = (correctos / total_ejemplos) * 100
print(f"La Precisión (Accuracy) del modelo en Test es: {accuracy:.2f}%")

X_prueba_bin = torch.from_numpy(X_test_buzz[:15]).float().to(device)
y_real_bin = y_test_buzz[:15]

with torch.no_grad():
     clasificaciones = (model_bin(X_prueba_bin) >= 0.5).float().cpu().numpy()

print("\n--- Analizando 15 Predicciones Individuales ---")
for i in range(15):
    txt_real = "Popular (1)" if y_real_bin[i][0] == 1 else "No popular (0)"
    txt_pred = "Popular (1)" if clasificaciones[i][0] == 1 else "No popular (0)"
    flag = "✅" if y_real_bin[i][0] == clasificaciones[i][0] else "❌"
    print(f"Test {i+1} | Real: {txt_real} | Predicho: {txt_pred} {flag}")

### 2.5 Visualización de la Función Sigmoide
Para demostrar conceptualmente cómo la red toma decisiones, graficaremos la función Sigmoide. Esto muestra cómo cualquier salida de la red ($z$) es comprimida al rango probabilístico de $0$ a $1$, con un umbral de decisión en $0.5$.

In [ ]:
# Recreamos exactamente el gráfico de la teoría para la Sigmoide
z = np.linspace(-10, 10, 100)
# Fórmula de la sigmoide: 1 / (1 + e^-z)
g_z = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 5))
# Graficamos la curva sigmoide en rojo
plt.plot(z, g_z, color='red', linewidth=3, label=r'$g(z) = \frac{1}{1 + e^{-z}}$')

# Añadimos la línea del umbral de decisión en 0.5 (negra)
plt.axhline(y=0.5, color='black', linestyle='-', label='Umbral (0.5)')
# Añadimos una tenue línea punteada vertical en el 0
plt.axvline(x=0, color='gray', linestyle=':')

# Configuraciones visuales del gráfico
plt.title('Función de Activación Sigmoide')
plt.xlabel('z (Salida numérica interna de la red)')
plt.ylabel('g(z) (Probabilidad)')
plt.yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
plt.grid(alpha=0.3)
plt.legend()
plt.show()